In [1]:
!python --version

Python 3.11.14


## GDC

https://docs.gdc.cancer.gov/API/Users_Guide/Downloading_Files/

sample  
 ├── primary_site       (organ)  
 ├── disease_type       (broad class)  
 ├── subtype_global     (cross-cancer)  
 ├── subtype_tissue     (within-cancer)  
 └── histology          (biological family)  

### TCGA / GDC Portal

https://portal.gdc.cancer.gov/

### 🧠 Big picture (this is powerful)

- You’ve essentially built the backbone of:
- cBioPortal-like explorer
- cohort stratification engine
- patient-specific pipeline (your perturbation_agent 🔥)

### 💡 Flow

> project → project_id  
> primary_sites → pid and disease_type  
> subtypes → subtype_id  
> stages →  stage_id  
> case_id → case_ids or UUIDs  
> samples → sample_id [tumor, normal]  
> aliquots → aliquot_id  
> files → file_id  


In [3]:
import os, sys
import pandas as pd

from pathlib import Path

ROOT = Path().resolve().parent.parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

from libs.calc_degs_lib import CALC_DEGS
from libs.tcga_gdc_lib import *
from libs.Basic import *


ROOT: /home/flavio/uv
SRC added: /home/flavio/uv/src


### 👉 a TCGA cohort builder + data retrieval engine

Which means you can easily extend to:


> pan-cancer analyses  
> patient-specific pipelines (your perturb_agent 🔥)  
> automated workflows (Nextflow-ready)  

#### 🚀 Next steps

> Production pipeline  
  - CLI tool
  - cached queries
  - reproducible outputs

> Expression matrix builder  
  - auto-download
  - merge TSVs
  - DESeq2-ready matrix

> Patient-level analysis
  - per-case DEG
  - pathway perturbation scoring

### Defaults

In [4]:
ROOT = Path().resolve().parent
root_data = os.path.join(ROOT, "data/tcga")

gdc = GDC(root_data=root_data)

os.listdir(root_data)[:10]


["samples_for_TCGA-BRCA_subtype_'lobular'_tumor_'other'_subtype_tissue_'breast_lobular'.tsv",
 'subtype_for_TCGA-KIRC.tsv',
 'degs.txt',
 'cases_for_TCGA-LUSC.tsv',
 "samples_for_TCGA-ACC_subtype_'sarcoma'_tumor_'sarcoma'_subtype_tissue_'sarcoma'.tsv",
 'subtype_for_TCGA-HNSC.tsv',
 'cases_for_TCGA-PCPG.tsv',
 'Gene_Expression_Quantification_for_TCGA-BRCA_case_0045349c-69d9-4306-a403-c9c1fa836644_sample_type_Primary_Tumor_stage_unknown_fileid_0e0df72c-33c0-4e4f-939c-a4d45a6e1ea3.tsv',
 "samples_for_TCGA-LUAD_subtype_'other'_tumor_'lymphoma'_subtype_tissue_'other'.tsv",
 "samples_for_TCGA-LUAD_subtype_'clear_cell'_tumor_'other'_subtype_tissue_'clear_cell'.tsv"]

### Get all programs

In [5]:
force=False
verbose=True

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)


File read at '/home/flavio/uv/perturb_agent/data/tcga/programs.txt'


In [6]:
np.array(prog_list)

array(['TCGA', 'MATCH', 'TARGET', 'CGCI', 'CMI', 'APOLLO', 'BEATAML1.0',
       'CPTAC', 'MP2PRT', 'ALCHEMIST', 'CCDI', 'CCG', 'CDDP_EAGLE',
       'CTSP', 'EXCEPTIONAL_RESPONDERS', 'FM', 'HCMI', 'MMRF', 'NCICCR',
       'OHSU', 'ORGANOID', 'RC', 'REBC', 'TRIO', 'VAREPOP', 'WCDT'],
      dtype='<U22')

### Primary sites given a program

In [7]:
gdc.url_gdc_project

'https://api.gdc.cancer.gov/projects'

In [8]:
force=False
verbose=True

program = 'TCGA'

df_psi = gdc.get_primary_sites(program=program, force=force, verbose=verbose)

print(len(df_psi))
df_psi.head(12)


Table opened ((33, 5)) at '/home/flavio/uv/perturb_agent/data/tcga/primary_site_program_TCGA.tsv'
33


,pid,primary_site,project_id,disease_type,name
0,TCGA-ACC,Adrenal gland,TCGA-ACC,Adenomas and Adenocarcinomas,Adrenocortical Carcinoma
1,TCGA-PCPG,"Adrenal gland, Retroperitoneum and peritoneum,...",TCGA-PCPG,Paragangliomas and Glomus Tumors,Pheochromocytoma and Paraganglioma
2,TCGA-BLCA,Bladder,TCGA-BLCA,"Epithelial Neoplasms, NOS, Squamous Cell Neopl...",Bladder Urothelial Carcinoma
3,TCGA-LGG,Brain,TCGA-LGG,Gliomas,Brain Lower Grade Glioma
4,TCGA-GBM,Brain,TCGA-GBM,"Not Reported, Gliomas",Glioblastoma Multiforme
5,TCGA-BRCA,Breast,TCGA-BRCA,"Adnexal and Skin Appendage Neoplasms, Adenomas...",Breast Invasive Carcinoma
6,TCGA-LUAD,Bronchus and lung,TCGA-LUAD,"Ductal and Lobular Neoplasms, Cystic, Mucinous...",Lung Adenocarcinoma
7,TCGA-LUSC,Bronchus and lung,TCGA-LUSC,"Squamous Cell Neoplasms, Adenomas and Adenocar...",Lung Squamous Cell Carcinoma
8,TCGA-MESO,"Bronchus and lung, Heart, mediastinum, and pleura",TCGA-MESO,Mesothelial Neoplasms,Mesothelioma
9,TCGA-CESC,"Cervix uteri, Ovary",TCGA-CESC,"Squamous Cell Neoplasms, Cystic, Mucinous and ...",Cervical Squamous Cell Carcinoma and Endocervi...


In [9]:
df_psi.tail(3)

,pid,primary_site,project_id,disease_type,name
30,TCGA-THCA,Thyroid gland,TCGA-THCA,"Squamous Cell Neoplasms, Epithelial Neoplasms,...",Thyroid Carcinoma
31,TCGA-UCS,"Uterus, NOS, Corpus uteri",TCGA-UCS,"Basal Cell Neoplasms, Complex Mixed and Stroma...",Uterine Carcinosarcoma
32,TCGA-UCEC,"Uterus, NOS, Corpus uteri",TCGA-UCEC,"Not Reported, Cystic, Mucinous and Serous Neop...",Uterine Corpus Endometrial Carcinoma


### Subtypes explanation

👉 GDC does NOT give you clean biological subtypes
👉 You must derive them yourself

This is actually a feature, not a bug — because:

> you control granularity  
> you can harmonize across cancers  
> you avoid noisy labels  


💡 If you want to level this up

I can help you build:

🔬 A cross-TCGA subtype harmonizer
maps all projects → unified subtype ontology
handles synonyms (adenocarcinoma, NOS, etc.)
outputs clean ML-ready labels

👉 That would massively improve your perturbation analysis.

In [10]:
ipsi = 6
pid = df_psi.iloc[ipsi].pid
primary_site = df_psi.iloc[ipsi].primary_site

print(ipsi, pid, primary_site)

6 TCGA-LUAD Bronchus and lung


- Brain tumors do NOT use AJCC staging (3 - TCGA-LGG Brain)

In [11]:
force=False
verbose=True

pid = df_psi.iloc[ipsi].pid
primary_site = df_psi.iloc[ipsi].primary_site

print(ipsi, pid, primary_site)

df_cases, df_subt, df_prof = gdc.get_cases_and_subtypes(pid=pid, batch_size=200, do_filter=False, force=force, verbose=verbose)

print("df_cases", len(df_cases), "df_subt", len(df_subt))

6 TCGA-LUAD Bronchus and lung
Table opened ((585, 24)) at '/home/flavio/uv/perturb_agent/data/tcga/cases_for_TCGA-LUAD.tsv'
df_cases 585 df_subt 23


### df_subt

In [12]:
print(len(df_subt))
df_subt

23


,pid,subtype_global,tumor_class,subtype_tissue,stage,n
0,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,unknown,136
1,TCGA-LUAD,other,other,other,unknown,136
2,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IA,76
3,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IB,71
4,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IIIA,40
5,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IIB,37
6,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IIA,27
7,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IV,14
8,TCGA-LUAD,other,other,other,Stage IB,9
9,TCGA-LUAD,other,other,other,Stage IA,8


### df_cases

In [13]:
df_cases.head(3)

,primary_site,disease_type,case_id,diagnoses,pid,subtype_global,stage_ajcc,primary_diagnosis,tumor_grade,stage_clin,...,disease_type_norm,diagnosis_norm,tumor_class,histology,subtype_tissue,consistency,validity,n,frac,sstage
0,Bronchus and lung,Adenomas and Adenocarcinomas,07b5663f-9a54-4462-b6c1-6fc8116b8714,"[{'primary_diagnosis': 'Adenocarcinoma, NOS', ...",TCGA-LUAD,adenocarcinoma_generic,Stage IA,"Adenocarcinoma, NOS",NaN,NaN,...,adenomas and adenocarcinomas,adenocarcinoma,adenocarcinoma,epithelial,lung_adenocarcinoma,ok,valid,1,0.001709,I
1,Bronchus and lung,Adenomas and Adenocarcinomas,294ff941-aea1-4588-9a0e-e9f5393e2bb6,[{'primary_diagnosis': 'Infiltrating duct carc...,TCGA-LUAD,other,Stage IA,"Infiltrating duct carcinoma, NOS",NaN,NaN,...,adenomas and adenocarcinomas,infiltrating duct carcinoma,other,other,other,ok,ambiguous,1,0.001709,I
2,Bronchus and lung,Adenomas and Adenocarcinomas,ebb33753-9d38-4368-9033-3e55f129d00d,[{'primary_diagnosis': 'Adenocarcinoma with mi...,TCGA-LUAD,adenocarcinoma_generic,unknown,Adenocarcinoma with mixed subtypes,NaN,NaN,...,adenomas and adenocarcinomas,adenocarcinoma with mixed subtypes,adenocarcinoma,epithelial,lung_adenocarcinoma,ok,valid,1,0.001709,missing


In [14]:
cols = ["pid", "case_id", "subtype_global", "tumor_class", "subtype_tissue", "stage"]

df_cases[cols].head(12)

,pid,case_id,subtype_global,tumor_class,subtype_tissue,stage
0,TCGA-LUAD,07b5663f-9a54-4462-b6c1-6fc8116b8714,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IA
1,TCGA-LUAD,294ff941-aea1-4588-9a0e-e9f5393e2bb6,other,other,other,Stage IA
2,TCGA-LUAD,ebb33753-9d38-4368-9033-3e55f129d00d,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,unknown
3,TCGA-LUAD,781f40c9-c099-4c96-8269-ebe2a449c93d,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IIB
4,TCGA-LUAD,5134c56f-8286-4ec8-8348-237cee7dad5e,other,other,other,Stage IIA
5,TCGA-LUAD,369f14c4-2191-4962-a309-3e23ddc4e5fc,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IIA
6,TCGA-LUAD,e51f717f-72e8-400e-9f35-fc5dc32fdf71,other,other,other,unknown
7,TCGA-LUAD,bbe88801-34f3-46d2-bbfd-b46c3901ed71,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IB
8,TCGA-LUAD,d2c1e896-6886-4122-bb48-5fbcd3f641f4,other,other,other,unknown
9,TCGA-LUAD,108a71cf-b9db-47cd-aa74-c03ec989b41b,other,other,other,Stage IIA


### Samples

In [15]:
force=False
force=False
verbose=True

isubt=0

row = df_subt.iloc[isubt]

pid = row.pid
subtype_global = row.subtype_global
tumor_class = row.tumor_class
subtype_tissue = row.subtype_tissue

s_case = f"{pid} subtype '{subtype_global}' tumor '{tumor_class}' subtype_tissue '{subtype_tissue}'"
s_case = title_replace(s_case)

# print(s_case)
df_samples = gdc.get_samples_for_pid_subtypes(pid=pid, subtype_global=subtype_global,
                                             tumor_class=tumor_class, subtype_tissue=subtype_tissue, s_case=s_case,
                                             batch_size=200, force=force, verbose=verbose)

print(len(df_samples))



Table opened ((585, 24)) at '/home/flavio/uv/perturb_agent/data/tcga/cases_for_TCGA-LUAD.tsv'
>>> TCGA-LUAD_subtype_'adenocarcinoma_generic'_tumor_'adenocarcinoma'_subtype_tissue_'lung_adenocarcinoma'
Table opened ((403, 14)) at '/home/flavio/uv/perturb_agent/data/tcga/samples_for_TCGA-LUAD_subtype_'adenocarcinoma_generic'_tumor_'adenocarcinoma'_subtype_tissue_'lung_adenocarcinoma'.tsv'
403


### Available sample types

In [16]:
df_samples.sample_type.unique()

array(['Solid Tissue Normal', 'Primary Tumor', 'Blood Derived Normal'],
      dtype=object)

In [17]:
df2 = df_samples[~df_samples.sample_type.str.contains('Blood', case=False, na=False)]
print(len(df_samples), len(df2))
df2.head()

403 278


,case_id,submitter_id,sample_id,sample_type,barcode_id,file_id,file_name,data_type,data_format,pid,subtype_global,tumor_class,subtype_tissue,stage
0,07b5663f-9a54-4462-b6c1-6fc8116b8714,TCGA-44-2655,298662ad-dbff-4344-9b80-d744c258c292,Solid Tissue Normal,TCGA-44-2655-11A,8f5ab8b4-72a8-4365-a54a-f7dac0e384bd,3211c900-d9ba-4b44-9329-8bc476c59266.methylati...,Methylation Beta Value,TXT,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IA
1,07b5663f-9a54-4462-b6c1-6fc8116b8714,TCGA-44-2655,9fcabda1-ea79-4188-8b3f-7d0fd060a819,Primary Tumor,TCGA-44-2655-01A,b2b7acb4-ece5-4e00-bd05-2de5f985bb8a,deae5c6f-6617-4729-98b2-b89ee95cf663.somatic.S...,Structural Rearrangement,BEDPE,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IA
2,07b5663f-9a54-4462-b6c1-6fc8116b8714,TCGA-44-2655,9fcabda1-ea79-4188-8b3f-7d0fd060a819,Primary Tumor,TCGA-44-2655-01A,87ff6237-1814-4f3b-abd8-d0e0d6f472be,92b25422-ea11-4b60-bc35-e86c0d73db9f.mirbase21...,Isoform Expression Quantification,TXT,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IA
3,07b5663f-9a54-4462-b6c1-6fc8116b8714,TCGA-44-2655,9fcabda1-ea79-4188-8b3f-7d0fd060a819,Primary Tumor,TCGA-44-2655-01A,5c628172-65f9-4bd7-b608-c24f6745f5ab,af6115c8-f05b-43ea-9e95-5e9cc78e2e89.somatic.S...,Structural Rearrangement,BEDPE,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IA
4,07b5663f-9a54-4462-b6c1-6fc8116b8714,TCGA-44-2655,9fcabda1-ea79-4188-8b3f-7d0fd060a819,Primary Tumor,TCGA-44-2655-01A,6773903f-ed74-4bb1-8fe4-583ef441861b,HILLY_p_TCGA_b90_wRedos_SNP_N_GenomeWideSNP_6_...,Simple Germline Variation,TSV,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IA


In [18]:
df3 = df_samples[df_samples.sample_type == 'Primary Tumor']
print(len(df_samples), len(df2), len(df3))
df3.head()

403 278 252


,case_id,submitter_id,sample_id,sample_type,barcode_id,file_id,file_name,data_type,data_format,pid,subtype_global,tumor_class,subtype_tissue,stage
1,07b5663f-9a54-4462-b6c1-6fc8116b8714,TCGA-44-2655,9fcabda1-ea79-4188-8b3f-7d0fd060a819,Primary Tumor,TCGA-44-2655-01A,b2b7acb4-ece5-4e00-bd05-2de5f985bb8a,deae5c6f-6617-4729-98b2-b89ee95cf663.somatic.S...,Structural Rearrangement,BEDPE,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IA
2,07b5663f-9a54-4462-b6c1-6fc8116b8714,TCGA-44-2655,9fcabda1-ea79-4188-8b3f-7d0fd060a819,Primary Tumor,TCGA-44-2655-01A,87ff6237-1814-4f3b-abd8-d0e0d6f472be,92b25422-ea11-4b60-bc35-e86c0d73db9f.mirbase21...,Isoform Expression Quantification,TXT,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IA
3,07b5663f-9a54-4462-b6c1-6fc8116b8714,TCGA-44-2655,9fcabda1-ea79-4188-8b3f-7d0fd060a819,Primary Tumor,TCGA-44-2655-01A,5c628172-65f9-4bd7-b608-c24f6745f5ab,af6115c8-f05b-43ea-9e95-5e9cc78e2e89.somatic.S...,Structural Rearrangement,BEDPE,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IA
4,07b5663f-9a54-4462-b6c1-6fc8116b8714,TCGA-44-2655,9fcabda1-ea79-4188-8b3f-7d0fd060a819,Primary Tumor,TCGA-44-2655-01A,6773903f-ed74-4bb1-8fe4-583ef441861b,HILLY_p_TCGA_b90_wRedos_SNP_N_GenomeWideSNP_6_...,Simple Germline Variation,TSV,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IA
5,07b5663f-9a54-4462-b6c1-6fc8116b8714,TCGA-44-2655,9fcabda1-ea79-4188-8b3f-7d0fd060a819,Primary Tumor,TCGA-44-2655-01A,e256a770-8d9c-4fc7-87ea-4f986948407a,HILLY_p_TCGA_b90_wRedos_SNP_N_GenomeWideSNP_6_...,Copy Number Segment,TXT,TCGA-LUAD,adenocarcinoma_generic,adenocarcinoma,lung_adenocarcinoma,Stage IA


### CBioPortal studies

In [19]:
study_list = gdc.cbioportal_studies()

study_list.sort()


lista = [x for x in study_list if 'luad' in x]

print("\n".join(lista))

luad_broad
luad_cas_2020
luad_cptac_2020
luad_cptac_gdc
luad_msk_npjpo_2021
luad_mskcc_2015
luad_mskcc_2020
luad_mskcc_2023_met_organotropism
luad_mskimpact_2021
luad_oncosg_2020
luad_tcga
luad_tcga_gdc
luad_tcga_pan_can_atlas_2018
luad_tcga_pub
luad_tsp


### GDC barcodes ~ submitter

In [20]:
barcodes = list(np.unique(df3.barcode_id))
len(barcodes)

10

In [21]:
# if len(barcodes[0].split('-')[-1]) == 3:
#    barcodes = [x[:-1] for x in barcodes]
barcodes

['TCGA-44-2655-01A',
 'TCGA-62-A471-01A',
 'TCGA-62-A471-01Z',
 'TCGA-67-3773-01A',
 'TCGA-67-3773-01Z',
 'TCGA-86-6851-01A',
 'TCGA-86-6851-01Z',
 'TCGA-91-6848-01A',
 'TCGA-NJ-A7XG-01A',
 'TCGA-NJ-A7XG-01Z']

In [22]:
pid

'TCGA-LUAD'

In [23]:
# mat = pid.lower().split('-')
# study_id = mat[1] + '_' + mat[0]
# print(">>>", study_id)

In [24]:
barcodes

['TCGA-44-2655-01A',
 'TCGA-62-A471-01A',
 'TCGA-62-A471-01Z',
 'TCGA-67-3773-01A',
 'TCGA-67-3773-01Z',
 'TCGA-86-6851-01A',
 'TCGA-86-6851-01Z',
 'TCGA-91-6848-01A',
 'TCGA-NJ-A7XG-01A',
 'TCGA-NJ-A7XG-01Z']

### get_mutations_from_samples() - must prepar samples_ids and study_id

In [25]:
study_id=pid
sample_ids=barcodes

if study_id[0].isupper():
	mat = study_id.lower().split('-')
	# cBioPortal disease - tcga 
	study_id = mat[1] + '_' + mat[0]

if study_id == "luad_tcga":
	study_id = "luad_tcga_pan_can_atlas_2018"


print(">>", study_id)

sample_ids = [str(x).strip() for x in sample_ids if str(x).strip()]
if not sample_ids:
	raise ValueError("sample_ids is empty.")

if len(sample_ids[0].split('-')[-1]) == 3:
	sample_ids = [x[:-1] for x in sample_ids]

print(sample_ids)


verbose=True

df_mut = gdc.get_mutations_from_samples(sample_ids=sample_ids, study_id=study_id, verbose=verbose)
len(df_mut)

>> luad_tcga_pan_can_atlas_2018
['TCGA-44-2655-01', 'TCGA-62-A471-01', 'TCGA-62-A471-01', 'TCGA-67-3773-01', 'TCGA-67-3773-01', 'TCGA-86-6851-01', 'TCGA-86-6851-01', 'TCGA-91-6848-01', 'TCGA-NJ-A7XG-01', 'TCGA-NJ-A7XG-01']
>>> https://www.cbioportal.org/api/molecular-profiles/luad_tcga_pan_can_atlas_2018_mutations/mutations/fetch


1773

In [26]:
df_mut.head(2).T

,0,1
sample_id,TCGA-44-2655-01,TCGA-44-2655-01
barcode,TCGA-44-2655,TCGA-44-2655
pid,luad_tcga_pan_can_atlas_2018,luad_tcga_pan_can_atlas_2018
mol_profile_id,luad_tcga_pan_can_atlas_2018_mutations,luad_tcga_pan_can_atlas_2018_mutations
symbol,CD1D,CEACAM7
refseq_mrna_id,NM_001766.3,NM_006890.3
entrez_gene_id,912,1087
protein_mut,L263I,G70V
mutation_type,Missense_Mutation,Missense_Mutation
mutation_status,.,.


In [27]:
study_id=pid
sample_ids=barcodes

force=True
verbose=True


dff, df_mut = gdc.get_df_mut_transform_mutation_table(study_id=study_id, s_case=s_case, sample_ids=sample_ids, 
													  session=None, timeout=60, force=force, verbose=verbose)

dff.head(8)

>>> TCGA-LUAD_subtype_'adenocarcinoma_generic'_tumor_'adenocarcinoma'_subtype_tissue_'lung_adenocarcinoma' // luad_tcga_pan_can_atlas_2018 len = 10 - ['TCGA-44-2655-01', 'TCGA-62-A471-01', 'TCGA-62-A471-01', 'TCGA-67-3773-01', 'TCGA-67-3773-01']...
>>> https://www.cbioportal.org/api/molecular-profiles/luad_tcga_pan_can_atlas_2018_mutations/mutations/fetch
Table saved ((1749, 11)) at '/home/flavio/uv/perturb_agent/data/tcga/mutations_summ_for_study_TCGA-LUAD_subtype_'adenocarcinoma_generic'_tumor_'adenocarcinoma'_subtype_tissue_'lung_adenocarcinoma'_samples_TCGA-44-2655-01_to_TCGA-62-A471-01.tsv'
Table saved ((1773, 27)) at '/home/flavio/uv/perturb_agent/data/tcga/mutations_anal_for_study_TCGA-LUAD_subtype_'adenocarcinoma_generic'_tumor_'adenocarcinoma'_subtype_tissue_'lung_adenocarcinoma'_samples_TCGA-44-2655-01_to_TCGA-62-A471-01.tsv'


,pid,sample_id,barcode,symbol,refseq_mrna_id,entrez_gene_id,protein_mut,mutation_type,variant_type,chr,n_mutations
0,luad_tcga_pan_can_atlas_2018,TCGA-44-2655-01,TCGA-44-2655,ABCB4,NM_000443.3,5244,K1258N,Missense_Mutation,SNP,7,1
1,luad_tcga_pan_can_atlas_2018,TCGA-44-2655-01,TCGA-44-2655,ACTL7B,NM_006686.3,10880,M41I,Missense_Mutation,SNP,9,1
2,luad_tcga_pan_can_atlas_2018,TCGA-44-2655-01,TCGA-44-2655,ADAM30,NM_021794.3,11085,Q67H,Missense_Mutation,SNP,1,1
3,luad_tcga_pan_can_atlas_2018,TCGA-44-2655-01,TCGA-44-2655,AHNAK2,NM_138420.2,113146,D2351E,Missense_Mutation,SNP,14,1
4,luad_tcga_pan_can_atlas_2018,TCGA-44-2655-01,TCGA-44-2655,ALDH5A1,NM_170740.1,7915,A198P,Missense_Mutation,SNP,6,1
5,luad_tcga_pan_can_atlas_2018,TCGA-44-2655-01,TCGA-44-2655,ALMS1P1,NA,200420,X103_splice,Splice_Site,SNP,2,1
6,luad_tcga_pan_can_atlas_2018,TCGA-44-2655-01,TCGA-44-2655,ANAPC11,NM_001002248.1,51529,D43Y,Missense_Mutation,SNP,17,1
7,luad_tcga_pan_can_atlas_2018,TCGA-44-2655-01,TCGA-44-2655,ANKEF1,NM_198798.1,63926,Y459C,Missense_Mutation,SNP,20,1


### Development & tests

In [ ]:
import requests

sample_ids = barcodes
study_id = pid

if study_id[0].isupper():
	mat = study_id.lower().split('-')
	# cBioPortal disease - tcga 
	study_id = mat[1] + '_' + mat[0]

if study_id == "luad_tcga":
	study_id = "luad_tcga_pan_can_atlas_2018"


print(">>", study_id)

sample_ids = [str(x).strip() for x in sample_ids if str(x).strip()]
if not sample_ids:
	raise ValueError("sample_ids is empty.")

if len(sample_ids[0].split('-')[-1]) == 3:
	sample_ids = [x[:-1] for x in sample_ids]

print(sample_ids)


session = requests.Session()
timeout = 60
verbose = True

molecular_profile_id = f"{study_id}_mutations"

http = session or requests.Session()

url = f"{gdc.url_cbioportal}/molecular-profiles/{molecular_profile_id}/mutations/fetch"
print(">>>", url)

payload = {
	"sampleIds": sample_ids
}

headers = {
	"Accept": "application/json",
	"Content-Type": "application/json",
}

resp = http.post(url, json=payload, headers=headers, timeout=timeout)

# Helpful error message from cBioPortal
if not resp.ok:
	msg = ""
	try:
		msg = resp.json()
	except Exception:
		msg = resp.text

	raise RuntimeError(
		f"cBioPortal request failed: HTTP {resp.status_code} | "
		f"profile={molecular_profile_id} | details={msg}"
	)

data = resp.json()
df = pd.DataFrame(data)
df.head(2).T

In [ ]:
df.columns

In [ ]:
cols = ['sampleId', 'patientId', 'studyId', 'molecularProfileId',
       'entrezGeneId', 'keyword', 'proteinChange', 'mutationType',
       'mutationStatus', 'center', 'tumorRefCount', 'normalAltCount',
       'normalRefCount', 'variantType', 'chr', 'startPosition',
       'endPosition', 'referenceAllele', 'uniqueSampleKey',
       'uniquePatientKey', 'validationStatus', 'tumorAltCount',
       'ncbiBuild', 'variantAllele', 'refseqMrnaId', 'proteinPosStart',
       'proteinPosEnd']

# the selected cols + others not listed
# cols = [c for c in cols if c in df.columns.to_list()] + [c for c in df.columns if c not in cols]
df = df[cols]

np.array(cols)


df['keyword'] = [x.split(' ')[0] if isinstance(x, str) else x for x in df['keyword']]


rename_cols = ['sampleId', 'barcode', 'pid', 'mol_profile_id',
       'entrez_gene_id', 'symbol', 'protein_mut', 'mutation_type',
       'mutation_status', 'center', 'tumorRefCount', 'normalAltCount',
       'normalRefCount', 'variant_type', 'chr', 'start',
       'end', 'ref_allele', 'unique_sample_key',
       'unique_patient_key', 'validation_status', 'tumor_alt_count',
       'ncbi_build', 'variant_allele', 'refseq_mrna_id', 'protein_pos_start', 'protein_pos_end']


df.columns = rename_cols

df.head(2).T

In [ ]:
order_cols = ['sampleId', 'barcode', 'pid', 'mol_profile_id',
              'symbol', 'refseq_mrna_id', 'entrez_gene_id', 
              'protein_mut', 'mutation_type', 'mutation_status',
              'ref_allele', 'variant_allele', 'variant_type', 
              'chr', 'start', 'end', 
              'validation_status', 'protein_pos_start', 'protein_pos_end', 'tumor_alt_count',
       'ncbi_build', 
       'center', 'tumorRefCount', 'normalAltCount', 'normalRefCount', 'unique_sample_key',
       'unique_patient_key']

df = df[order_cols]
df.head(2).T